# Fine-tune Qwen2.5-1.5B-Instruct with Alpaca Dataset using LlamaFactory

This example demonstrates how to fine-tune Qwen2.5-1.5B-Instruct model with the Alpaca Dataset
using LlamaFactory `BuiltinTrainer` from Kubeflow Trainer SDK.

This notebook walks you through the prerequisites and how to submit a TrainJob to bootstrap
the LoRA SFT fine-tuning workflow with LlamaFactory.

Qwen2.5-1.5B-Instruct: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

Alpaca Dataset: https://huggingface.co/datasets/tatsu-lab/alpaca

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Prerequisites

### Install Official Training Runtimes

You need to make sure that you've installed the Kubeflow Trainer Controller Manager and Kubeflow Training Runtimes mentioned in the [installation guide](https://www.kubeflow.org/docs/components/trainer/operator-guides/installation/).

Also ensure the `llamafactory-lora-sft` ClusterTrainingRuntime is deployed in your cluster.

In [ ]:
# List all available Kubeflow Training Runtimes.
from kubeflow.trainer import *
from kubeflow.trainer.constants import constants

client = TrainerClient()
for runtime in client.list_runtimes():
    print(runtime)

## Define LlamaFactory Training Configuration

LlamaFactory uses a YAML-based configuration. We pass the training parameters as a Python
dictionary via `LlamaFactoryConfig`. The SDK will automatically create a ConfigMap from this
configuration and mount it into the trainer container.

In [ ]:
# LlamaFactory training configuration for LoRA SFT.
llamafactory_config = {
    ### model
    "model_name_or_path": "/workspace/model",
    "trust_remote_code": True,

    ### method
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "lora_rank": 8,
    "lora_target": "all",

    ### dataset
    "dataset_dir": "/workspace/dataset",
    "dataset": "alpaca_en_demo",
    "template": "default",
    "cutoff_len": 1024,
    "max_samples": 500,
    "preprocessing_num_workers": 4,

    ### output
    "output_dir": "/workspace/output",
    "logging_steps": 10,
    "save_steps": 500,
    "overwrite_output_dir": True,

    ### train
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 1.0e-4,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "bf16": True,
    "ddp_timeout": 180000000,
}

## Bootstrap LLM Fine-tuning Workflow

Kubeflow TrainJob will train the model in the referenced ClusterTrainingRuntime.
The SDK creates a ConfigMap from the `LlamaFactoryConfig` and submits the TrainJob.

In [ ]:
job_name = client.train(
    runtime="llamafactory-lora-sft",
    initializer=Initializer(
        dataset=HuggingFaceDatasetInitializer(
            storage_uri="hf://tatsu-lab/alpaca"
        ),
        model=HuggingFaceModelInitializer(
            storage_uri="hf://Qwen/Qwen2.5-1.5B-Instruct",
        ),
    ),
    trainer=BuiltinTrainer(
        config=LlamaFactoryConfig(
            config=llamafactory_config,
            num_nodes=2,
            resources_per_node={
                "gpu": 1,
            },
        ),
    ),
)

## Wait for Running Status

In [ ]:
client.wait_for_job_status(name=job_name, status={"Running"})

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

### Dataset Initializer

In [ ]:
for line in client.get_job_logs(job_name, follow=True, step=constants.DATASET_INITIALIZER):
    print(line)

### Model Initializer

In [ ]:
for line in client.get_job_logs(job_name, follow=True, step=constants.MODEL_INITIALIZER):
    print(line)

### Trainer Node

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

for line in client.get_job_logs(job_name, follow=True):
    print(line)

## Verify TrainJob Completion

Check the TrainJob status to ensure it completed successfully.

In [ ]:
client.wait_for_job_status(name=job_name, timeout=20)

## Get the Fine-tuned Model

After the Trainer node completes the fine-tuning task, the fine-tuned model will be stored
into the `/workspace/output` directory, which can be shared across Pods through PVC mounting.
You can find it in another Pod's `/<mountDir>/output` directory if you mount the PVC under
`/<mountDir>`.